# 深度 Q 学习 - 月球着陆器

在本作业中，你将训练一个智能体，使月球着陆器安全降落在月球表面的着陆区。


# 大纲
- [ 1 - 导入软件包 <img align="Right" src="./images/lunar_lander.gif" width = 60% >](#1)
- [ 2 - 超参数](#2)
- [ 3 - 月球着陆器环境](#3)
  - [ 3.1 动作空间](#3.1)
  - [ 3.2 观测空间](#3.2)
  - [ 3.3 奖励](#3.3)
  - [ 3.4 回合终止](#3.4)
- [ 4 - 加载环境](#4)
- [ 5 - 与 Gym 环境交互](#5)
    - [ 5.1 探索环境动态](#5.1)
- [ 6 - 深度 Q 学习](#6)
  - [ 6.1 目标网络](#6.1)
    - [ 练习 1](#ex01)
  - [ 6.2 经验回放](#6.2)
- [ 7 - 带经验回放的深度 Q 学习算法](#7)
  - [ 练习 2](#ex02)
- [ 8 - 更新网络权重](#8)
- [ 9 - 训练智能体](#9)
- [ 10 - 查看训练后的智能体如何行动](#10)
- [ 11 - 恭喜！](#11)
- [ 12 - 参考资料](#12)

<a name="1"></a>
## 1 - 导入软件包

我们将使用以下软件包：
- `numpy` 是用于 Python 科学计算的软件包。
- `deque` 将作为记忆缓冲区的数据结构。
- `namedtuple` 将用于存储经验元组。
- `gym` 工具包包含一系列可用于测试强化学习算法的环境。需要注意，本 notebook 使用的是 `gym` `0.24.0` 版本。
- 渲染月球着陆器环境需要 `PIL.Image` 和 `pyvirtualdisplay`。
- 我们将使用 `tensorflow.keras` 框架中的若干模块来构建深度学习模型。
- `utils` 是一个包含本作业辅助函数的模块。你无需修改此文件中的代码。

运行下面的单元格以导入所有必要的软件包。

In [ ]:
import time
from collections import deque, namedtuple

import gym
import numpy as np
import PIL.Image
import tensorflow as tf
import utils

from pyvirtualdisplay import Display
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.losses import MSE
from tensorflow.keras.optimizers import Adam

In [ ]:
# Set up a virtual display to render the Lunar Lander environment.
Display(visible=0, size=(840, 480)).start();

# Set the random seed for TensorFlow
tf.random.set_seed(utils.SEED)

<a name="2"></a>
## 2 - 超参数

运行下面的单元格以设置超参数。

In [ ]:
MEMORY_SIZE = 100_000     # size of memory buffer
GAMMA = 0.995             # discount factor
ALPHA = 1e-3              # learning rate  
NUM_STEPS_FOR_UPDATE = 4  # perform a learning update every C time steps

<a name="3"></a>
## 3 - 月球着陆器环境

在本 notebook 中，我们将使用 [OpenAI Gym 库](https://www.gymlibrary.ml/)。Gym 库提供了多种强化学习环境。简单来说，环境表示一个需要解决的问题或任务。在本 notebook 中，我们将尝试使用强化学习解决月球着陆器环境。

月球着陆器环境的目标是让月球着陆器安全降落在月球表面的着陆区。着陆区由两根旗杆标识，始终位于坐标 `(0,0)`，但着陆器也可以降落在着陆区之外。着陆器从环境顶部中央开始，其质心会受到一个随机初始力，并拥有无限燃料。如果获得 `200` 分，则认为该环境已解决。

<br>
<br>
<figure>
  <img src = "images/lunar_lander.gif" width = 40%>
      <figcaption style = "text-align: center; font-style: italic">图 1. 月球着陆器环境。</figcaption>
</figure>



<a name="3.1"></a>
### 3.1 动作空间

智能体有四种可用的离散动作：

* 不执行任何操作。
* 点燃右侧发动机。
* 点燃主发动机。
* 点燃左侧发动机。

每个动作都有一个对应的数值：

```python
Do nothing = 0
Fire right engine = 1
Fire main engine = 2
Fire left engine = 3
```

<a name="3.2"></a>
### 3.2 观测空间

智能体的观测空间由包含 8 个变量的状态向量组成：

* 它的 $(x,y)$ 坐标。着陆区始终位于坐标 $(0,0)$。
* 它的线速度 $(\dot x,\dot y)$。
* 它的角度 $\theta$。
* 它的角速度 $\dot \theta$。
* 两个布尔值 $l$ 和 $r$，分别表示两条着陆腿是否与地面接触。

<a name="3.3"></a>
### 3.3 奖励

月球着陆器环境采用以下奖励机制：

* 降落在着陆区并完全停稳，可获得约 100～140 分。
* 如果着陆器远离着陆区，会损失奖励。
* 如果着陆器坠毁，会获得 -100 分。
* 如果着陆器完全停稳，会获得 +100 分。
* 每条着陆腿与地面接触可获得 +10 分。
* 每帧点燃主发动机会获得 -0.3 分。
* 每帧点燃侧发动机会获得 -0.03 分。

<a name="3.4"></a>
### 3.4 回合终止

出现以下情况时，回合结束（即环境进入终止状态）：

* 月球着陆器坠毁（即着陆器主体与月球表面接触）。

* 着陆器的 $x$ 坐标大于 1。

你可以查看 [OpenAI Gym 文档](https://www.gymlibrary.ml/environments/box2d/lunar_lander/)，了解该环境的完整说明。

<a name="4"></a>
## 4 - 加载环境

首先，使用 `.make()` 方法从 `gym` 库加载 `LunarLander-v2` 环境。`LunarLander-v2` 是月球着陆器环境的最新版本，可以在 [OpenAI Gym 文档](https://www.gymlibrary.ml/environments/box2d/lunar_lander/#version-history)中阅读其版本历史。

In [ ]:
env = gym.make('LunarLander-v2')

加载环境后，我们使用 `.reset()` 方法将环境重置为初始状态。着陆器从环境顶部中央开始，我们可以使用 `.render()` 方法渲染环境的第一帧。

In [ ]:
env.reset()
PIL.Image.fromarray(env.render(mode='rgb_array'))

为了稍后构建神经网络，需要知道状态向量的大小和有效动作的数量。可以分别使用环境的 `.observation_space.shape` 和 `action_space.n` 方法获取这些信息。

In [ ]:
state_size = env.observation_space.shape
num_actions = env.action_space.n

print('State Shape:', state_size)
print('Number of actions:', num_actions)

<a name="5"></a>
## 5——与 Gym 环境交互

Gym 库实现了标准的“智能体—环境循环”形式：

<br>
<center>
<video src = "./videos/rl_formalism.m4v" width="840" height="480" controls autoplay loop poster="./images/rl_formalism.png"> </video>
<figcaption style = "text-align:center; font-style:italic">图 2：智能体—环境循环形式。</figcaption>
</center>
<br>

在标准的“智能体—环境循环”形式中，智能体以离散时间步 $t=0,1,2,...$ 与环境交互。在每个时间步 $t$，智能体根据对环境状态 $S_t$ 的观察，使用策略 $\pi$ 选择动作 $A_t$。智能体获得数值奖励 $R_t$，并在下一个时间步转移到新状态 $S_{t+1}$。

<a name="5.1"></a>
### 5.1 探索环境的动态过程

在 OpenAI 的 Gym 环境中，我们使用 `.step()` 方法运行环境动态过程的一个时间步。在本实验所用的 `gym` 版本中，`.step()` 方法接收一个动作并返回四个值：

* `observation`（**对象**）：表示环境观察结果的环境特定对象。在月球着陆器环境中，它对应一个 NumPy 数组，其中包含 [3.2 观察空间](#3.2) 一节所述的着陆器位置和速度。


* `reward`（**浮点数**）：执行给定动作后返回的奖励数值。在月球着陆器环境中，它对应 [3.3 奖励](#3.3) 一节所述的 `numpy.float64` 类型浮点数。


* `done`（**布尔值**）：当 done 为 `True` 时，表示本回合已经终止，需要重置环境。


* `info`（**字典**）：有助于调试的诊断信息。本 Notebook 不会使用该变量，但为完整起见在此列出。

要开始一个回合，需要将环境重置为初始状态。为此，我们使用 `.reset()` 方法。

In [ ]:
# Reset the environment and get the initial state.
initial_state = env.reset()

环境重置后，智能体可以使用 `.step()` 方法开始在环境中执行动作。请注意，智能体每个时间步只能执行一个动作。

在下面的单元格中，可以选择不同的动作，并观察返回值如何随所执行动作而变化。请记住，在此环境中，智能体有四个离散动作可用，我们在代码中使用对应的数值来指定它们：

```python
Do nothing = 0
Fire right engine = 1
Fire main engine = 2
Fire left engine = 3
```

In [ ]:
# Select an action
action = 0

# Run a single time step of the environment's dynamics with the given action.
next_state, reward, done, info = env.step(action)

with np.printoptions(formatter={'float': '{:.3f}'.format}):
    print("Initial State:", initial_state)
    print("Action:", action)
    print("Next State:", next_state)
    print("Reward Received:", reward)
    print("Episode Terminated:", done)
    print("Info:", info)

在实践中，训练智能体时会使用循环，使智能体能够在一个回合中连续执行许多动作。

<a name="6"></a>
## 6 - 深度 Q 学习

当状态空间和动作空间均为离散空间时，可以使用贝尔曼方程迭代估计动作价值函数：

$$
Q_{i+1}(s,a) = R + \gamma \max_{a'}Q_i(s',a')
$$

当 $i\to\infty$ 时，这种迭代方法会收敛到最优动作价值函数 $Q^*(s,a)$。这意味着，智能体只需逐步探索状态-动作空间，并持续更新对 $Q(s,a)$ 的估计，直到它收敛到最优动作价值函数 $Q^*(s,a)$。然而，当状态空间为连续空间时，探索整个状态-动作空间在实践中是不可能的。因此，在实践中也无法逐步估计 $Q(s,a)$，使其收敛到 $Q^*(s,a)$。

在深度 $Q$ 学习中，我们使用神经网络估计动作价值函数 $Q(s,a)\approx Q^*(s,a)$，从而解决这一问题。我们将这个神经网络称为 $Q$ 网络，可以在每次迭代中调整其权重，通过最小化贝尔曼方程中的均方误差对其进行训练。

遗憾的是，事实证明，在强化学习中使用神经网络估计动作价值函数非常不稳定。幸运的是，可以采用几种技术来避免不稳定。这些技术包括使用***目标网络***和***经验回放***。我们将在后续各节中探索这两种技术。

<a name="6.1"></a>
### 6.1 目标网络

我们可以在每次迭代时调整 $Q$ 网络的权重，使贝尔曼方程中的均方误差最小。其目标值为：

$$
y = R + \gamma \max_{a'}Q(s',a';w)
$$

其中，$w$ 是 $Q$ 网络的权重。这意味着我们在每次迭代时调整权重 $w$，以最小化以下误差：

$$
\overbrace{\underbrace{R + \gamma \max_{a'}Q(s',a'; w)}_{\rm {y~target}} - Q(s,a;w)}^{\rm {Error}}
$$

请注意，这会带来一个问题，因为 $y$ 目标在每次迭代中都会改变。目标不断移动可能导致振荡和不稳定。为了避免这种情况，可以创建一个单独的神经网络来生成 $y$ 目标。我们将这个独立的神经网络称为**目标 $\hat Q$ 网络**，其架构与原始 $Q$ 网络相同。使用目标 $\hat Q$ 网络后，上面的误差变为：

$$
\overbrace{\underbrace{R + \gamma \max_{a'}\hat{Q}(s',a'; w^-)}_{\rm {y~target}} - Q(s,a;w)}^{\rm {Error}}
$$

其中，$w^-$ 和 $w$ 分别是目标 $\hat Q$ 网络和 $Q$ 网络的权重。

在实践中，我们将使用以下算法：每经过 $C$ 个时间步，就使用 $\hat Q$ 网络生成 $y$ 目标，并使用 $Q$ 网络的权重更新目标 $\hat Q$ 网络的权重。我们将使用**软更新**来更新目标 $\hat Q$ 网络的权重 $w^-$。这意味着将按以下规则更新权重 $w^-$：
 
$$
w^-\leftarrow \tau w + (1 - \tau) w^-
$$

其中，$\tau\ll 1$。通过使用软更新，可以确保目标值 $y$ 缓慢变化，从而显著提高学习算法的稳定性。

<a name="ex01"></a>
### 练习 1

在本练习中，你将创建 $Q$ 网络和目标 $\hat Q$ 网络，并设置优化器。请记住，深度 $Q$ 网络（DQN）是一个近似动作价值函数 $Q(s,a)\approx Q^*(s,a)$ 的神经网络，它通过学习如何将状态映射到 $Q$ 值来实现这一点。

为了解决月球着陆器环境，我们将采用具有以下架构的 DQN：

* 一个以 `state_size` 为输入的 `Input` 层。

* 一个包含 `64` 个单元并使用 `relu` 激活函数的 `Dense` 层。

* 一个包含 `64` 个单元并使用 `relu` 激活函数的 `Dense` 层。

* 一个包含 `num_actions` 个单元并使用 `linear` 激活函数的 `Dense` 层。这将是网络的输出层。


在下面的单元格中，应使用上述模型架构创建 $Q$ 网络和目标 $\hat Q$ 网络。请记住，$Q$ 网络和目标 $\hat Q$ 网络具有相同的架构。

最后，应将 `Adam` 设置为优化器，学习率等于 `ALPHA`。回想一下，`ALPHA` 已在[超参数](#2)一节中定义。请注意，本练习应使用已导入的软件包：
```python
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
```

In [ ]:
# UNQ_C1
# GRADED CELL

# Create the Q-Network
q_network = Sequential([
    ### START CODE HERE ### 

    ### END CODE HERE ### 
    ])

# Create the target Q^-Network
target_q_network = Sequential([
    ### START CODE HERE ### 

    ### END CODE HERE ###
    ])

### START CODE HERE ### 
optimizer = None
### END CODE HERE ###

In [ ]:
# UNIT TEST
from public_tests import *

test_network(q_network)
test_network(target_q_network)
test_optimizer(optimizer, ALPHA) 

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
```python
# Create the Q-Network
q_network = Sequential([
    Input(shape=state_size),                      
    Dense(units=64, activation='relu'),            
    Dense(units=64, activation='relu'),            
    Dense(units=num_actions, activation='linear'),
    ])

# Create the target Q^-Network
target_q_network = Sequential([
    Input(shape=state_size),                       
    Dense(units=64, activation='relu'),            
    Dense(units=64, activation='relu'),            
    Dense(units=num_actions, activation='linear'), 
    ])

optimizer = Adam(learning_rate=ALPHA)                                  
``` 

<a name="6.2"></a>
### 6.2 经验回放

当智能体与环境交互时，其经历的状态、动作和奖励本质上是连续的。如果智能体尝试从这些连续经验中学习，可能会因经验之间存在强相关性而遇到问题。为避免这种情况，我们采用一种称为**经验回放**的技术，为训练智能体生成不相关的经验。经验回放会将智能体的经验（即智能体接收的状态、动作和奖励）存储在记忆缓冲区中，然后从缓冲区随机采样一个经验小批量来执行学习。智能体与环境交互时，每个时间步都会将经验元组 $(S_t, A_t, R_t, S_{t+1})$ 添加到记忆缓冲区。

为方便起见，我们将经验存储为命名元组。

In [ ]:
# Store experiences as named tuples
experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])

通过使用经验回放，可以避免有问题的相关性、振荡和不稳定性。此外，经验回放还允许智能体在多次权重更新中重复使用同一条经验，从而提高数据效率。

<a name="7"></a>
## 7 - 使用经验回放的深度 Q 学习算法

现在我们已经了解了将要使用的所有技术，可以将它们组合起来，得到使用经验回放的深度 Q 学习算法。
<br>
<br>
<figure>
  <img src = "images/deep_q_algorithm.png" width = 90% style = "border: thin silver solid; padding: 0px">
      <figcaption style = "text-align: center; font-style: italic">图 3：使用经验回放的深度 Q 学习。</figcaption>
</figure>

<a name="ex02"></a>
### 练习 2

在本练习中，你将实现上面*图 3* 所列算法的第 ***12*** 行，还要计算 $y$ 目标值与 $Q(s,a)$ 值之间的损失。在下面的单元格中，通过将 $y$ 目标值设为以下内容来完成 `compute_loss` 函数：

$$
\begin{equation}
    y_j =
    \begin{cases}
      R_j & \text{if episode terminates at step  } j+1\\
      R_j + \gamma \max_{a'}\hat{Q}(s_{j+1},a') & \text{otherwise}\\
    \end{cases}       
\end{equation}
$$

需要注意以下几点：

* `compute_loss` 函数接收一个经验元组小批次。该经验元组小批次会被解包，以提取 `states`、`actions`、`rewards`、`next_states` 和 `done_vals`。请记住，这些变量是 *TensorFlow 张量*，其大小取决于小批次大小。例如，如果小批次大小为 `64`，那么 `rewards` 和 `done_vals` 都将是包含 `64` 个元素的 TensorFlow 张量。


* 当变量是包含多个元素的张量时，无法使用 `if/else` 语句设置 $y$ 目标值。但请注意，可以使用 `done_vals` 在一行代码中实现上述操作。为此，回想一下，`done` 变量是布尔变量：当某个回合在第 $j+1$ 步终止时，其值为 `True`，否则为 `False`。考虑到布尔值 `True` 的数值为 `1`，布尔值 `False` 的数值为 `0`，可以使用因子 `(1 - done_vals)` 在一行代码中实现上述操作。提示：请注意，当 `done_vals` 为 `True` 时，`(1 - done_vals)` 的值为 `0`；当 `done_vals` 为 `False` 时，其值为 `1`。

最后，通过计算 `y_targets` 与 `q_values` 之间的均方误差（`MSE`）来计算损失。计算均方误差时，应使用已经导入的软件包 `MSE`：
```python
from tensorflow.keras.losses import MSE
```

In [ ]:
# UNQ_C2
# GRADED FUNCTION: calculate_loss

def compute_loss(experiences, gamma, q_network, target_q_network):
    """ 
    Calculates the loss.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
      q_network: (tf.keras.Sequential) Keras model for predicting the q_values
      target_q_network: (tf.keras.Sequential) Karas model for predicting the targets
          
    Returns:
      loss: (TensorFlow Tensor(shape=(0,), dtype=int32)) the Mean-Squared Error between
            the y targets and the Q(s,a) values.
    """
    
    # Unpack the mini-batch of experience tuples
    states, actions, rewards, next_states, done_vals = experiences
    
    # Compute max Q^(s,a)
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)
    
    # Set y = R if episode terminates, otherwise set y = R + γ max Q^(s,a).
    ### START CODE HERE ### 
    y_targets = None
    ### END CODE HERE ###
    
    # Get the q_values
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]),
                                                tf.cast(actions, tf.int32)], axis=1))
        
    # Compute the loss
    ### START CODE HERE ### 
    loss = None 
    ### END CODE HERE ### 
    
    return loss

In [ ]:
# UNIT TEST    
test_compute_loss(compute_loss)

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
```python
def compute_loss(experiences, gamma, q_network, target_q_network):
    """ 
    Calculates the loss.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
      q_network: (tf.keras.Sequential) Keras model for predicting the q_values
      target_q_network: (tf.keras.Sequential) Karas model for predicting the targets
          
    Returns:
      loss: (TensorFlow Tensor(shape=(0,), dtype=int32)) the Mean-Squared Error between
            the y targets and the Q(s,a) values.
    """

    
    # Unpack the mini-batch of experience tuples
    states, actions, rewards, next_states, done_vals = experiences
    
    # Compute max Q^(s,a)
    max_qsa = tf.reduce_max(target_q_network(next_states), axis=-1)
    
    # Set y = R if episode terminates, otherwise set y = R + γ max Q^(s,a).
    y_targets = rewards + (gamma * max_qsa * (1 - done_vals))
    
    # Get the q_values
    q_values = q_network(states)
    q_values = tf.gather_nd(q_values, tf.stack([tf.range(q_values.shape[0]),
                                                tf.cast(actions, tf.int32)], axis=1))
    
    # Calculate the loss
    loss = MSE(y_targets, q_values)
    
    return loss

``` 
    

<a name="8"></a>
## 8 - 更新网络权重

我们将使用下面的 `agent_learn` 函数实现[图 3](#7) 所列算法的第 ***12–14*** 行。`agent_learn` 函数会使用自定义训练循环更新 $Q$ 网络和目标 $\hat Q$ 网络的权重。由于使用自定义训练循环，需要通过 `tf.GradientTape` 实例获取梯度，然后调用 `optimizer.apply_gradients()` 更新 $Q$ 网络的权重。请注意，我们还使用 `@tf.function` 装饰器来提高性能。若不使用此装饰器，训练时间会增加一倍。如果想进一步了解如何使用 `@tf.function` 提高性能，请参阅 [TensorFlow 文档](https://www.tensorflow.org/guide/function)。

该函数的最后一行使用[软更新](#6.1)更新目标 $\hat Q$ 网络的权重。如果想了解其代码实现，建议查看 `utils` 模块中的 `utils.update_target_network` 函数。

In [ ]:
@tf.function
def agent_learn(experiences, gamma):
    """
    Updates the weights of the Q networks.
    
    Args:
      experiences: (tuple) tuple of ["state", "action", "reward", "next_state", "done"] namedtuples
      gamma: (float) The discount factor.
    
    """
    
    # Calculate the loss
    with tf.GradientTape() as tape:
        loss = compute_loss(experiences, gamma, q_network, target_q_network)

    # Get the gradients of the loss with respect to the weights.
    gradients = tape.gradient(loss, q_network.trainable_variables)
    
    # Update the weights of the q_network.
    optimizer.apply_gradients(zip(gradients, q_network.trainable_variables))

    # update the weights of target q_network
    utils.update_target_network(q_network, target_q_network)

<a name="9"></a>
## 9 - 训练智能体

现在，我们已经准备好训练智能体来解决月球着陆器环境。在下面的单元格中，我们将逐行实现 [图 3](#7) 中的算法（请注意，为方便参考，我们已在下方包含了同一算法。这样你就无需在 Notebook 中上下滚动）：

* **第 1 行**：我们初始化容量为 $N =$ `MEMORY_SIZE` 的 `memory_buffer`。请注意，我们使用 `deque` 作为 `memory_buffer` 的数据结构。


* **第 2 行**：由于我们已经在[练习 1](#ex01) 中初始化了 `q_network`，因此跳过此行。


* **第 3 行**：我们初始化 `target_q_network`，将其权重设置为与 `q_network` 的权重相同。


* **第 4 行**：我们开始外层循环。请注意，我们已将 $M =$ `num_episodes = 2000`。这个数值是合理的，因为使用本 Notebook 的默认参数，智能体应当能够在少于 `2000` 个回合内解决月球着陆器环境。


* **第 5 行**：我们使用 `.reset()` 方法将环境重置为初始状态，并获取初始状态。


* **第 6 行**：我们开始内层循环。请注意，我们已将 $T =$ `max_num_timesteps = 1000`。这意味着，如果回合在 `1000` 个时间步后仍未终止，它将自动终止。


* **第 7 行**：智能体观察当前 `state`，并使用 $\epsilon$-贪心策略选择一个 `action`。智能体一开始使用 $\epsilon =$ `epsilon = 1` 的值，由此得到的 $\epsilon$-贪心策略等价于等概率随机策略。这意味着在训练开始时，无论观察到什么 `state`，智能体都只会随机采取动作。随着训练进行，我们将使用给定的 $\epsilon$-衰减率，缓慢减小 $\epsilon$ 的值直至某个最小值。我们希望该最小值接近零，因为 $\epsilon = 0$ 的值会得到等价于贪心策略的 $\epsilon$-贪心策略。这意味着在训练接近结束时，智能体会倾向于选择它认为（根据过去的经验）能够最大化 $Q(s,a)$ 的 `action`。我们会将 $\epsilon$ 的最小值设为 `0.01`，而不是精确地设为 0，因为我们始终希望在训练期间保留少量探索。如果想了解代码中的实现方式，建议查看 `utils` 模块中的 `utils.get_action` 函数。


* **第 8 行**：我们使用 `.step()` 方法在环境中执行给定的 `action`，并获得 `reward` 和 `next_state`。


* **第 9 行**：我们将 `experience(state, action, reward, next_state, done)` 元组存入 `memory_buffer`。请注意，我们还会存储 `done` 变量，以便跟踪回合何时终止。这样，我们就能在[练习 2](#ex02) 中设置 $y$ 目标。


* **第 10 行**：我们检查是否满足执行学习更新的条件。为此，我们使用自定义的 `utils.check_update_conditions` 函数。该函数检查是否已经过 $C =$ `NUM_STEPS_FOR_UPDATE = 4` 个时间步，以及 `memory_buffer` 中是否有足够的经验元组来填满一个小批量。例如，如果小批量大小为 `64`，那么 `memory_buffer` 至少应包含 `64` 个经验元组，才能满足后一个条件。如果条件得到满足，`utils.check_update_conditions` 函数将返回 `True`；否则返回 `False`。


* **第 11 至 14 行**：如果 `update` 变量为 `True`，我们便执行一次学习更新。学习更新包括：从 `memory_buffer` 中随机采样一个经验元组小批量、设置 $y$ 目标、执行梯度下降，以及更新网络权重。我们将使用在[第 8 节](#8) 中定义的 `agent_learn` 函数来执行后三项操作。


* **第 15 行**：在内层循环每次迭代结束时，我们将 `next_state` 设为新的 `state`，以便循环从这个新状态重新开始。此外，我们还会检查回合是否已到达终止状态（即检查 `done = True`）。如果已经到达终止状态，就跳出内层循环。


* **第 16 行**：在外层循环每次迭代结束时，我们会更新 $\epsilon$ 的值，并检查环境是否已被解决。如果智能体在最近 `100` 个回合中获得的平均分达到 `200`，我们便认为环境已经解决。如果环境尚未解决，则继续外层循环并开始一个新回合。

最后需要说明的是，我们加入了一些额外变量，用于跟踪智能体在每个回合中获得的总分。这有助于我们判断智能体是否已经解决环境，也能让我们了解智能体在训练期间的表现。我们还使用 `time` 模块来测量训练耗时。

<br>
<br>
<figure>
  <img src = "images/deep_q_algorithm.png" width = 90% style = "border: thin silver solid; padding: 0px">
      <figcaption style = "text-align: center; font-style: italic">图 4. 使用经验回放的深度 Q 学习。</figcaption>
</figure>
<br>

**注意：** 使用本 Notebook 的默认参数时，运行以下单元格需要 10 到 15 分钟。

In [ ]:
start = time.time()

num_episodes = 2000
max_num_timesteps = 1000

total_point_history = []

num_p_av = 100    # number of total points to use for averaging
epsilon = 1.0     # initial ε value for ε-greedy policy

# Create a memory buffer D with capacity N
memory_buffer = deque(maxlen=MEMORY_SIZE)

# Set the target network weights equal to the Q-Network weights
target_q_network.set_weights(q_network.get_weights())

for i in range(num_episodes):
    
    # Reset the environment to the initial state and get the initial state
    state = env.reset()
    total_points = 0
    
    for t in range(max_num_timesteps):
        
        # From the current state S choose an action A using an ε-greedy policy
        state_qn = np.expand_dims(state, axis=0)  # state needs to be the right shape for the q_network
        q_values = q_network(state_qn)
        action = utils.get_action(q_values, epsilon)
        
        # Take action A and receive reward R and the next state S'
        next_state, reward, done, _ = env.step(action)
        
        # Store experience tuple (S,A,R,S') in the memory buffer.
        # We store the done variable as well for convenience.
        memory_buffer.append(experience(state, action, reward, next_state, done))
        
        # Only update the network every NUM_STEPS_FOR_UPDATE time steps.
        update = utils.check_update_conditions(t, NUM_STEPS_FOR_UPDATE, memory_buffer)
        
        if update:
            # Sample random mini-batch of experience tuples (S,A,R,S') from D
            experiences = utils.get_experiences(memory_buffer)
            
            # Set the y targets, perform a gradient descent step,
            # and update the network weights.
            agent_learn(experiences, GAMMA)
        
        state = next_state.copy()
        total_points += reward
        
        if done:
            break
            
    total_point_history.append(total_points)
    av_latest_points = np.mean(total_point_history[-num_p_av:])
    
    # Update the ε value
    epsilon = utils.get_new_eps(epsilon)

    print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}", end="")

    if (i+1) % num_p_av == 0:
        print(f"\rEpisode {i+1} | Total point average of the last {num_p_av} episodes: {av_latest_points:.2f}")

    # We will consider that the environment is solved if we get an
    # average of 200 points in the last 100 episodes.
    if av_latest_points >= 200.0:
        print(f"\n\nEnvironment solved in {i+1} episodes!")
        q_network.save('lunar_lander_model.h5')
        break
        
tot_time = time.time() - start

print(f"\nTotal Runtime: {tot_time:.2f} s ({(tot_time/60):.2f} min)")

我们可以绘制得分历史记录，查看智能体在训练期间的进步。

In [ ]:
# Plot the point history
utils.plot_history(total_point_history)

<a name="10"></a>
## 10 - 观察训练后的智能体实际运行

现在智能体已经训练完成，我们可以观察它的实际表现。我们将使用 `utils.create_video` 函数创建一段视频，展示智能体如何使用训练后的 $Q$ 网络与环境交互。`utils.create_video` 函数使用 `imageio` 库创建视频。该库会产生一些可能分散注意力的警告，因此我们运行下面的代码来抑制这些警告。

In [ ]:
# Suppress warnings from imageio
import logging
logging.getLogger().setLevel(logging.ERROR)

在下面的单元格中，我们会创建一段视频，展示智能体如何使用训练后的 `q_network` 与月球着陆器环境交互。视频以给定的 `filename` 保存到 `videos` 文件夹。我们使用 `utils.embed_mp4` 函数将视频嵌入 Jupyter Notebook，这样无需下载即可直接在此处观看。

需要注意的是，由于月球着陆器开始时会有一个随机初始力施加在其质心上，因此每次运行下面的单元格都会看到不同的视频。如果智能体训练得当，无论施加在质心上的初始力如何，它每次都应能够让月球着陆器降落在着陆区内。

In [ ]:
filename = "./videos/lunar_lander.mp4"

utils.create_video(filename, env, q_network)
utils.embed_mp4(filename)

<a name="11"></a>
## 11 - 恭喜！

你已成功使用带经验回放的深度 Q 学习，训练智能体将月球着陆器安全降落在月球表面的着陆区。恭喜！

<a name="12"></a>
## 12 - 参考资料

如果你想进一步了解深度 Q 学习，我们建议阅读以下论文。


* [通过深度强化学习实现人类水平的控制](https://storage.googleapis.com/deepmind-media/dqn/DQNNaturePaper.pdf)


* [使用深度强化学习进行连续控制](https://arxiv.org/pdf/1509.02971.pdf)


* [使用深度强化学习玩 Atari 游戏](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf)